# Crop classification per plot, satellite imagery

Predicts the crop type of a plot (**oil palm, cocoa, coffee, cassava, forest, other**)
from Sentinel-2 time series.

## Data

The notebook runs **right away on a simulated dataset** with the real structure, so that the
pipeline is validated end to end. As soon as you have your real data, put it in
`data/parcelles.parquet`: the notebook will use it automatically, with nothing else to change.

### Verified real sources (August 2026)

| Class | Source | Link |
|---|---|---|
| Oil palm | Descals et al. 2024, 18,812 labelled validation points + 10 m rasters, CC BY 4.0 | `10.5281/zenodo.13379129` |
| Cocoa | COLD-CI, cocoa / non-cocoa labelled polygons, Côte d'Ivoire | arXiv 2606.20767 |
| Cocoa (map) | Kalischek et al., *Nature Food* 2023, 10 m map of Côte d'Ivoire/Ghana | github.com/D1noFuzi/cocoamapping |
| Annual crops | WorldCereal Reference Data Module (cereal-oriented) | esa-worldcereal.org/en/reference-data |

⚠️ `data/` is excluded from git: neither the farmers' personal data nor the rasters are committed.

## Expected format for your real data

One row per plot, in `data/parcelles.parquet`:

- `classe`: the label to predict (one of the values in `CLASSES`)
- `bloc`: identifier of the plot's geographic area (**essential**, see below)
- `B2_01` … `B12_12`: mean reflectance of the plot, one column per band and per month

The `bloc` field is used to split training / test **spatially**. Two neighbouring plots
look very much alike (same soil, same weather, same image date): with a classic random
split, nearly identical plots end up on both sides and the measured score is
overestimated. The comparison cell further down quantifies this gap.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DOSSIER = Path("data")
FICHIER_PARCELLES = DOSSIER / "parcelles.parquet"
FICHIER_MODELE = DOSSIER / "modele_cultures.joblib"

CLASSES = ["palmier_huile", "cacao", "cafe", "manioc", "foret", "autre"]
# Sentinel-2 optical, then Sentinel-1 radar (VV, VH). Radar carries the structure of the
# canopy: it is the only signal that separates palm and forest, both green and
# saturated in NDVI all year round. Bands missing from the file are reported then
# ignored, so that the notebook also runs on a purely optical extraction.
BANDES = ["B2", "B3", "B4", "B8", "B11", "B12", "VV", "VH"]
MOIS = list(range(1, 13))

ALEA = np.random.default_rng(42)

## Loading (or simulation)

The profiles below encode what really distinguishes these crops in remote sensing:
**closed-canopy perennials** (palm, cocoa, forest) keep a high, nearly constant NDVI
all year round, whereas **cassava** follows its crop cycle with a strong seasonal amplitude.
It is this amplitude, more than instantaneous reflectance, that carries most of the signal.

In [ ]:
# ndvi: mean level | amplitude: trough-to-peak over the year | pic: month of the maximum
# nir / swir: near-infrared and shortwave-infrared reflectance levels
PROFILS = {
    "palmier_huile": dict(ndvi=0.82, amplitude=0.04, pic=6, nir=0.34, swir=0.17),
    "cacao": dict(ndvi=0.76, amplitude=0.07, pic=8, nir=0.31, swir=0.16),
    "cafe": dict(ndvi=0.71, amplitude=0.11, pic=7, nir=0.30, swir=0.19),
    "manioc": dict(ndvi=0.55, amplitude=0.32, pic=9, nir=0.28, swir=0.24),
    "foret": dict(ndvi=0.87, amplitude=0.03, pic=5, nir=0.36, swir=0.13),
    "autre": dict(ndvi=0.30, amplitude=0.15, pic=8, nir=0.22, swir=0.30),
}


def simuler_parcelles(n_parcelles=1200, n_blocs=30):
    """Realistic synthetic dataset: each block is an area with its dominant crop."""
    classes = list(PROFILS)
    # Area effect: soil, rainfall and acquisition dates shared by a whole block.
    # It is what makes neighbouring plots look alike, and therefore what creates spatial leakage.
    effet_bloc = ALEA.normal(0, 0.04, size=n_blocs)
    # One crop dominates each block, but the others remain present as a minority, otherwise
    # a whole validation fold may contain no plot of a given class.
    compositions = []
    for numero in range(n_blocs):
        poids = np.full(len(classes), 0.45 / (len(classes) - 1))
        poids[numero % len(classes)] = 0.55
        compositions.append(poids)

    lignes = []
    for _ in range(n_parcelles):
        bloc = int(ALEA.integers(n_blocs))
        classe = str(ALEA.choice(classes, p=compositions[bloc]))
        profil = PROFILS[classe]
        # Area effect + plot-specific variability (plant age, density).
        decalage = effet_bloc[bloc] + ALEA.normal(0, 0.03)
        enr = {"classe": classe, "bloc": bloc}
        for mois in MOIS:
            saison = np.cos(2 * np.pi * (mois - profil["pic"]) / 12)
            ndvi = np.clip(
                profil["ndvi"] + decalage + profil["amplitude"] * saison / 2
                + ALEA.normal(0, 0.02),
                0.02, 0.95,
            )
            nir = np.clip(profil["nir"] + ALEA.normal(0, 0.03), 0.05, 0.60)
            rouge = nir * (1 - ndvi) / (1 + ndvi)
            swir = np.clip(profil["swir"] + ALEA.normal(0, 0.02), 0.02, 0.50)
            enr[f"B8_{mois:02d}"] = nir
            enr[f"B4_{mois:02d}"] = rouge
            enr[f"B3_{mois:02d}"] = rouge * 1.25 + 0.010
            enr[f"B2_{mois:02d}"] = rouge * 1.10 + 0.015
            enr[f"B11_{mois:02d}"] = swir
            enr[f"B12_{mois:02d}"] = swir * 0.70
        lignes.append(enr)
    return pd.DataFrame(lignes)


if FICHIER_PARCELLES.exists():
    parcelles = pd.read_parquet(FICHIER_PARCELLES)
    print(f"Real data: {FICHIER_PARCELLES}")
    manquantes = {"classe", "bloc"} - set(parcelles.columns)
    if manquantes:
        raise ValueError(f"Columns missing from the file: {sorted(manquantes)}")
else:
    parcelles = simuler_parcelles()
    print(
        "⚠️  SIMULATED DATA: none of the results below is valid in the field.\n"
        f"    Put your real data in {FICHIER_PARCELLES} for real training."
    )

print(f"\n{len(parcelles)} plots, {parcelles['bloc'].nunique()} blocks")
print(parcelles["classe"].value_counts().to_string())

In [ ]:
# The real classes are not those of the simulated dataset: read them back from the file.
# Without this, the confusion matrix comes out empty and the model is saved with a list
# of classes that matches nothing.
CLASSES = sorted(parcelles["classe"].unique())
print(f"Classes: {CLASSES}")

## Features

The 72 raw reflectances are not given to the model: **vegetation indices** and their
annual statistics are extracted from them, which are much more stable from one image to the next.

- **NDVI**: vegetation vigour
- **NDWI**: canopy water content
- **NBR**: canopy structure (separates perennials and annuals well)
- **VV - VH**: difference between the two radar polarisations, a signature of canopy structure.
  It is the only feature that separates palm and forest, provided the Earth Engine extraction
  supplied the Sentinel-1 bands.

In [ ]:
def _indice(df, bande_a, bande_b):
    """Normalised index (a - b) / (a + b), computed month by month."""
    return pd.DataFrame(
        {
            mois: (df[f"{bande_a}_{mois:02d}"] - df[f"{bande_b}_{mois:02d}"])
            / (df[f"{bande_a}_{mois:02d}"] + df[f"{bande_b}_{mois:02d}"])
            for mois in MOIS
        }
    )


def construire_variables(df):
    variables = pd.DataFrame(index=df.index)
    series = {
        "ndvi": _indice(df, "B8", "B4"),
        "ndwi": _indice(df, "B3", "B8"),
        "nbr": _indice(df, "B8", "B12"),
    }
    for nom, serie in series.items():
        variables[f"{nom}_moy"] = serie.mean(axis=1)
        variables[f"{nom}_ecart_type"] = serie.std(axis=1)
        variables[f"{nom}_min"] = serie.min(axis=1)
        variables[f"{nom}_max"] = serie.max(axis=1)
        # Seasonal amplitude: the main perennial / annual discriminant.
        variables[f"{nom}_amplitude"] = variables[f"{nom}_max"] - variables[f"{nom}_min"]
    variables["mois_pic_ndvi"] = series["ndvi"].idxmax(axis=1)

    # A purely optical extraction has neither VV nor VH. This is reported rather than silently
    # ignored: without radar, palm and forest remain inseparable and the score plateaus.
    disponibles = [bande for bande in BANDES if f"{bande}_01" in df.columns]
    absentes = [bande for bande in BANDES if bande not in disponibles]
    if absentes:
        print(f"⚠️  Bands missing from the file, ignored: {absentes}")

    for bande in disponibles:
        colonnes = [f"{bande}_{mois:02d}" for mois in MOIS]
        variables[f"{bande}_moy"] = df[colonnes].mean(axis=1)
        variables[f"{bande}_ecart_type"] = df[colonnes].std(axis=1)

    # VV - VH difference: the regular rows of a plantation return a radar echo
    # different from that of a natural forest, where optical data sees no difference.
    if "VV" in disponibles and "VH" in disponibles:
        ecart = pd.DataFrame(
            {mois: df[f"VV_{mois:02d}"] - df[f"VH_{mois:02d}"] for mois in MOIS}
        )
        variables["vv_vh_moy"] = ecart.mean(axis=1)
        variables["vv_vh_ecart_type"] = ecart.std(axis=1)

    return variables


X = construire_variables(parcelles)
y = parcelles["classe"]
blocs = parcelles["bloc"]

print(f"{X.shape[1]} features built for {X.shape[0]} plots")
X.head()

## Random vs spatial split

The next cell measures the same model both ways. The gap between the two scores is the
share of optimism due to spatial leakage: it is the difference between the number reported and
the one that will be obtained on a new area.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GroupKFold, StratifiedGroupKFold, StratifiedKFold,
                                     cross_val_score)

# Whole blocks on one side of the split, while keeping all six classes
# represented in each fold.
DECOUPAGE_SPATIAL = StratifiedGroupKFold(5, shuffle=True, random_state=0)

modele_test = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)

score_aleatoire = cross_val_score(
    modele_test, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="f1_macro"
)
score_spatial = cross_val_score(
    modele_test, X, y, cv=DECOUPAGE_SPATIAL, groups=blocs, scoring="f1_macro"
)

print(f"Random split  : F1 = {score_aleatoire.mean():.3f} (± {score_aleatoire.std():.3f})")
print(f"Spatial split : F1 = {score_spatial.mean():.3f} (± {score_spatial.std():.3f})")
print(f"\nOptimism of the random split: {score_aleatoire.mean() - score_spatial.mean():+.3f}")
print("The spatial score is the one to keep and report.")

## Model comparison

Evaluated with spatial cross-validation only.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

candidats = {
    "Random forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1
    ),
    "Gradient boosting": HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.08, random_state=0
    ),
}

resultats = {}
for nom, candidat in candidats.items():
    scores = cross_val_score(candidat, X, y, cv=GroupKFold(5), groups=blocs, scoring="f1_macro")
    resultats[nom] = scores.mean()
    print(f"{nom:20s} F1 = {scores.mean():.3f} (± {scores.std():.3f})")

meilleur_nom = max(resultats, key=resultats.get)
modele = candidats[meilleur_nom]
print(f"\nSelected: {meilleur_nom}")

## Detailed evaluation

Training blocks and test blocks fully disjoint.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

app, test = next(GroupKFold(5).split(X, y, groups=blocs))
modele.fit(X.iloc[app], y.iloc[app])
predictions = modele.predict(X.iloc[test])

print(f"Training: {len(app)} plots / Test: {len(test)} plots (disjoint blocks)\n")
print(classification_report(y.iloc[test], predictions, zero_division=0))

figure, axes = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y.iloc[test], predictions, labels=CLASSES, normalize="true",
    values_format=".2f", cmap="Blues", xticks_rotation=45, ax=axes, colorbar=False,
)
axes.set_title("Confusion matrix (share per actual class)")
axes.set_xlabel("Predicted")
axes.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## Most useful features

Permutation importance, measured on the test set: unlike the trees' internal
importance, it does not overrate high-cardinality features.

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    modele, X.iloc[test], y.iloc[test],
    n_repeats=10, random_state=0, n_jobs=-1, scoring="f1_macro",
)
classement = (
    pd.Series(importance.importances_mean, index=X.columns)
    .sort_values(ascending=False)
    .head(15)
)

figure, axes = plt.subplots(figsize=(8, 5))
classement.sort_values().plot.barh(ax=axes, color="#2b7a4b")
axes.set_title("15 most discriminating features")
axes.set_xlabel("F1 loss when the feature is shuffled")
plt.tight_layout()
plt.show()

## Final model

Retrained on all available plots, then saved in `data/` (outside git).

In [ ]:
import joblib

modele.fit(X, y)
DOSSIER.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {"modele": modele, "variables": list(X.columns), "classes": CLASSES},
    FICHIER_MODELE,
)
print(f"Model saved: {FICHIER_MODELE}")

if not FICHIER_PARCELLES.exists():
    print(
        "\n⚠️  This model is trained on SIMULATED data."
        "\n    It must not be used to classify real plots."
    )

## Next step: plug in the real data

1. Download the palm validation points (Zenodo `10.5281/zenodo.13379129`,
   `Validation_points_GlobalOP2016-2021.zip`) and the COLD-CI cocoa polygons.
2. Extract for each point/polygon the monthly Sentinel-2 series over one year
   (Google Earth Engine, or `rasterio` + `geopandas` locally, these two packages are not
   installed yet).
3. Write the result to `data/parcelles.parquet` in the format described at the top of the notebook,
   filling `bloc` with a spatial grid (for example 10 km squares).
4. Rerun the notebook: it will switch to the real data on its own.

The `cafe` and `manioc` classes have no public labelled source for your area. They will
have to be built from your own field surveys: this is where the farmer list and their
plot codes (`organize_farmers.ipynb`) become useful again.